# LTX-2.5 vidax TPU micro-benchmark

Select **TPU v5e-8** in Kaggle Notebook Settings before running. This notebook intentionally stops unless it sees all eight TPU devices. Attach a Kaggle input dataset named `ltx25-checkpoints` with the three Hugging Face gated BF16 files under `LTX-2.5/` (the exact relative paths are checked below). Do not put a Hugging Face token in notebook source. This is a one-run timing/quality probe, not the 15-minute render.

The probe uses the unmodified official vidax benchmark harness, the LTX-2.5 distilled checkpoint, convolutional VAE, and TP=8. It writes one clip plus benchmark timings.


In [ ]:
import json, os, platform, shutil, subprocess, sys, time
from pathlib import Path
import jax

devices = jax.devices()
print(json.dumps({
    'python': sys.version, 'platform': platform.platform(),
    'jax': jax.__version__, 'devices': [{'platform': d.platform, 'kind': d.device_kind, 'id': d.id} for d in devices],
    'device_count': len(devices), 'working_free_bytes': shutil.disk_usage('/kaggle/working').free,
}, indent=2))
if len(devices) < 8 or any(d.platform != 'tpu' for d in devices):
    raise RuntimeError('TPU v5e-8 not allocated. Select it in Kaggle Settings → Accelerator, restart session, then rerun. No model download or compute was attempted.')


In [ ]:
# Check mounted weights first. The full checkpoint pack is roughly 70 GB, too large for the
# 20 GB /kaggle/working volume seen in earlier diagnostics, so keep it attached as an input dataset.
CKPT = Path('/kaggle/input/ltx25-checkpoints/LTX-2.5')
required = [
    CKPT / 'diffusion_models/ltx-2.5-22b-distilled-transformer-bf16.safetensors',
    CKPT / 'text_encoders/gemma4-12b-with-proj-ltx-2.5-bf16.safetensors',
    CKPT / 'vae/ltx-2.5-video-vae-conv-bf16.safetensors',
]
missing = [str(p) for p in required if not p.is_file()]
if missing:
    raise FileNotFoundError('Attach the gated LTX-2.5 BF16 checkpoint dataset as ltx25-checkpoints. Missing: ' + ', '.join(missing))
print(json.dumps([{'path':str(p), 'bytes':p.stat().st_size} for p in required], indent=2))


In [ ]:
REPO = Path('/kaggle/working/vidax')
if not REPO.exists():
    subprocess.run(['git','clone','https://github.com/FlyingGiraffe/vidax.git',str(REPO)], check=True)
subprocess.run([sys.executable,'-m','pip','install','-e','.[tpu]'], cwd=REPO, check=True)


In [ ]:
# One cold run using vidax's own compile-vs-generation harness, at the target 480p shape.
# This is a 1-run engineering probe, not a statistically stable benchmark.
import sys, time
from types import SimpleNamespace
sys.path.insert(0, str(REPO/'examples'))
sys.path.insert(0, str(REPO/'benchmarks'))
import generate_ltx2_5
import run_ltx2_5
import common

probe_args = SimpleNamespace(checkpoint_dir='/kaggle/input/ltx25-checkpoints', model_size='22B-distilled', task='t2v', vae_variant='conv', tensor_parallel_size=8)
gen_args = run_ltx2_5.build_args(probe_args)
gen_args.height = 480
gen_args.width = 832
gen_args.num_frames = 121
gen_args.fps = 24
gen_args.offload_dit_weights = False
gen_args.output_path = str(REPO/'out'/'ltx25_t2v_480p_8tp.mp4')
started = time.perf_counter()
benchmark_result = common.run_benchmark(model='ltx2_5', version='', size='22b_distilled_480p_8tp', task='t2v', main_fn=generate_ltx2_5.main, args=gen_args, num_runs=1)
wall = time.perf_counter() - started
print(json.dumps({'run':benchmark_result['runs'][0], 'device_kind':benchmark_result['device_kind'], 'device_count':benchmark_result['device_count'], 'resolution':benchmark_result['resolution'], 'num_frames':benchmark_result['num_frames'], 'tensor_parallel_size':benchmark_result['tensor_parallel_size'], 'benchmark_process_wall_seconds':round(wall,2)}, indent=2))


In [ ]:
# Verify the generated MP4, capture the benchmark JSON and create a contact sheet for manual lookdev.
import hashlib, glob
from IPython.display import Image, display
mp4s = sorted((REPO/'out').rglob('*.mp4'), key=lambda p:p.stat().st_mtime, reverse=True)
if not mp4s:
    raise FileNotFoundError('Benchmark reported success but no MP4 exists under vidax/out')
video = mp4s[0]
probe = subprocess.run(['ffprobe','-v','error','-show_entries','format=duration,size:stream=codec_name,width,height,r_frame_rate,nb_frames','-of','json',str(video)], check=True, text=True, stdout=subprocess.PIPE)
sheets = video.parent/'lookdev_contact.jpg'
subprocess.run(['ffmpeg','-y','-i',str(video),'-vf',"select='not(mod(n,30))',scale=384:-1,tile=4x2",'-frames:v','1',str(sheets)], check=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
bench_files = sorted((REPO/'benchmarks/results').glob('*.json'), key=lambda p:p.stat().st_mtime, reverse=True)
summary = {'status':'needs_visual_review','video':str(video),'bytes':video.stat().st_size,'sha256':hashlib.sha256(video.read_bytes()).hexdigest(),'ffprobe':json.loads(probe.stdout),'benchmark':benchmark_result['runs'][0],'benchmark_json':str(bench_files[0]) if bench_files else None,'benchmark_process_wall_seconds':round(wall,2),'contact_sheet':str(sheets)}
(REPO/'out'/'kaggle_tpu_ltx25_probe_summary.json').write_text(json.dumps(summary,indent=2))
print(json.dumps(summary,indent=2))
display(Image(filename=str(sheets)))
